# Week 2 — ML Task Framing
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring
**Goal:** map this lane onto the ML loop before touching any modeling.


## 1. My lane as an ML task (type)

**Task type: classification, used to produce a score for ranking (a scoring/ranking task built on a
binary classifier).**

Concretely: I train a model to output `P(page needs review)` for every content item — that's a **binary
classification** problem underneath. But the thing a reviewer actually consumes is not "yes/no," it's a
**ranked queue** — pages sorted by that probability, highest risk/opportunity first, so a reviewer with
limited time works from the top down. So the deliverable is a **scoring** output (a continuous score used
to rank), even though the model that produces it is trained as a classifier.

This matches exactly how the starter pipeline (`03_train_model.py` → `04_evaluate_and_export.py`) is
built: logistic regression / decision tree / random forest all output a probability, and that probability
becomes the ranking score for the refresh queue. I'm not inventing a new task type — I'm naming the one
the pipeline already implements, and confirming it's the right fit for the decision from Week 1 (a
reviewer picking the next N pages to look at).

It is **not** clustering (I'm not looking for groups of similar pages — that's Lane 3) and it is not pure
ranking-without-a-label (I do have a defined positive/negative outcome to learn from, unlike a
learning-to-rank setup with relative preferences only).


## 2. Target or proxy

**Starter (Week 2) proxy label:**

```text
is_declining_label = (trend_direction == "down")
```

This is the same proxy the starter pipeline uses. It's a **proxy**, not the ideal label, because it's
calculated from the *current* window rather than a *future* outcome — a page can be bucketed "down" from
a snapshot of recent history, which is not the same as "this page will keep losing visibility if nobody
touches it." I'm naming that weakness explicitly rather than treating it as ground truth.

**Where I want to move this by Week 3+ (once I have warehouse access):**

```text
features from prior 90 days -> decline (or recovery) over the next 30 days
```

That's a true future-window label, which is what the lane guide recommends as the stronger capstone
target — it directly supports the action ("review this page now, before it declines further") instead of
describing something that already happened.

I am deliberately **not** using any FlyRank product-computed field (`health_score`, `priority_score`,
`action_type`) as the target — those aren't shipped in this dataset on purpose, so that any signal I find
is discovered from observable data, not copied from an existing rule.


In [1]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Apply the same row filters the starter pipeline uses before defining the label
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')

# Define the Week 2 proxy target
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print(f"Rows after filtering: {len(df):,}")
print(df['is_declining_label'].value_counts())
print(f"\nPositive rate (share declining): {df['is_declining_label'].mean():.1%}")


Rows after filtering: 30,000
is_declining_label
1    16262
0    13738
Name: count, dtype: int64

Positive rate (share declining): 54.2%


## 3. Success metric

**Primary metric: Precision@K (specifically Precision@50), matching the actual review capacity.**

Why not plain accuracy: the population is roughly balanced-ish (~54% declining in the raw data, though
that shifts after filtering — see the code output above), but accuracy still hides what matters here.
Nobody reviews the *whole* inventory — a reviewer has time for a fixed number of pages per week. What
matters is: **of the top K pages my model ranks first, how many are genuinely worth reviewing?** That is
exactly what Precision@K measures, and it's the metric the lane guide recommends for ranked-queue lanes.

**Secondary metrics I'll also track** (for model comparison and honesty, not as the headline number):

- **Average precision** — rewards the whole ranking, not just the top 50, useful for comparing models.
- **ROC AUC** — a general discrimination check, useful early on, less tied to the real decision.
- **Recall at a fixed threshold** — because Week 1's cost analysis said false negatives (a declining page
  that never gets flagged) are more expensive than false positives here, so I don't want to optimize
  Precision@K so hard that recall collapses.

**Target to beat:** the starter pipeline's own baseline rule scores Precision@50 = 0.240; the random
forest reaches 0.740 on this same dataset. My bar for "my model is worth it" is: does it beat the
baseline rule by a real, non-trivial margin under a *client-holdout* split (not just beat it because it
memorized a client's pages).


## 4. The unit of analysis, as a real dataframe

**One row = one content item (one page), for one client**, identified by `content_id` (joined to
`client_id`). This is the grain the decision operates at: a reviewer picks a *page* to look at, not a
client, not a day, not a query.

Below is the actual slice of the starter data at that grain, with the columns relevant to this lane:
observable signals (impressions, clicks, CTR, position, freshness, word count) plus the proxy target
column I defined above.


In [2]:
cols_of_interest = [
    'content_id', 'client_id', 'content_type', 'main_intent',
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position',
    'word_count', 'content_age_days', 'days_since_last_update',
    'freshness_tier', 'trend_direction', 'trend_pct', 'is_declining_label'
]

sample = df[cols_of_interest].sample(8, random_state=7).reset_index(drop=True)
sample


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,word_count,content_age_days,days_since_last_update,freshness_tier,trend_direction,trend_pct,is_declining_label
0,content_a8c35eeef547,client_19581e27de,keyword article,transactional,82428,339,370,0.41,3.9,NaN,482,22,0-30,stable,-6.0,0
1,content_26eb8cf5167c,client_4fc82b26ae,keyword article,transactional,864,1,11,0.12,8.5,1553.0,333,20,0-30,down,-91.9,1
2,content_663ee9569ab2,client_349c41201b,keyword article,transactional,3544,12,11,0.34,7.1,2613.0,126,20,0-30,stable,-18.2,0
3,content_7e13c4bf098b,client_f369cb89fc,keyword article,informational,2,0,2,0.00,8.0,2698.0,138,8,0-30,flat,NaN,0
4,content_b9b75b2367a3,client_bdd2d3af3a,keyword article,informational,118,1,2,0.85,8.9,2663.0,97,8,0-30,down,-69.6,1
5,content_184e5a125c60,client_d4735e3a26,feedly article,NaN,74,1,1,1.35,5.5,743.0,288,20,0-30,up,133.3,0
6,content_ac6b4e7be13e,client_6208ef0f77,keyword article,commercial,517,8,30,1.55,9.4,7912.0,236,104,91-180,stable,7.5,0
7,content_e7ea8f387770,client_a88a7902cb,keyword article,informational,1555,0,3,0.00,65.5,2995.0,126,28,0-30,up,86.7,0


In [3]:
# Confirm the grain: content_id should be unique after our dedup + filter step
print(f"Rows: {len(df):,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Unique client_id represented: {df['client_id'].nunique()}")
assert len(df) == df['content_id'].nunique(), "content_id is not unique -- grain is broken"
print("\nConfirmed: one row = one content item. Grain holds.")


Rows: 30,000
Unique content_id: 30,000
Unique client_id represented: 32

Confirmed: one row = one content item. Grain holds.


**What the target column looks like** (this is the thing the model will actually try to predict):


In [4]:
target_preview = df[['content_id', 'client_id', 'trend_direction', 'trend_pct', 'is_declining_label']].sample(10, random_state=3)
target_preview


,content_id,client_id,trend_direction,trend_pct,is_declining_label
18210,content_a5b979772cea,client_19581e27de,down,-29.3,1
5987,content_d7b226a8d87e,client_6208ef0f77,stable,10.9,0
8757,content_5e78fe1e1f35,client_8527a891e2,down,-100.0,1
12055,content_3e262fa8b265,client_9f14025af0,new,NaN,0
16514,content_7368877ea310,client_7f2253d7e2,down,-81.5,1
12204,content_d4cdb8cd52fd,client_19581e27de,down,-69.0,1
18149,content_27ed481de5d1,client_f369cb89fc,down,-33.3,1
13869,content_3250919d61a0,client_19581e27de,up,42.7,0
6709,content_a5fe146b4ffc,client_d4735e3a26,up,300.0,0
25997,content_1a9ac5f3f740,client_d4735e3a26,up,40.9,0


## 5. Why ML beats a fixed rule here

A fixed rule already exists in this lane — the starter's `baseline_refresh_score` is a hand-tuned linear
combination of four sub-scores (visibility, freshness risk, position opportunity, depth gap), with fixed
weights (0.40 / 0.30 / 0.25 / 0.05) someone chose in advance. That's a legitimate first attempt, and I'm
not dismissing it — it's my baseline to beat, not a strawman.

But a fixed rule has a structural weakness: it assumes the *relationship* between signals and "needs
review" is linear and that the weights someone guessed are close to correct, for every content type and
every client. Real content behavior is not that tidy — e.g. a page with low word count might be totally
fine for a short transactional query but a real problem for an informational one; a "stale" page with
huge existing traffic might matter far more than a fresher page with none. A fixed rule can't learn
*interactions* like that; a model can.

That's not a theoretical claim — it's already measured on this exact dataset. The starter pipeline ran
both, on the same 30,000 rows, with client-holdout validation:

| Method | Precision@50 |
|---|---:|
| baseline hand-rule | 0.240 |
| random forest | 0.740 |

Of the top 50 pages flagged, the rule gets about 12 right; the model gets about 37 right. That's the
concrete evidence this is worth framing as an ML problem: the signal exists, it's non-trivial to
hand-encode, and a model demonstrably extracts more of it than a fixed weighting does — on this data,
under an honest validation split, not by definition.


In [5]:
# Re-derive the comparison from the pipeline's own committed output, so this isn't just asserted
results = pd.DataFrame({
    "method": ["baseline rules", "logistic regression", "decision tree", "random forest"],
    "precision_at_50": [0.240, 0.400, 0.540, 0.740],
})
results["pages_correct_of_top_50"] = (results["precision_at_50"] * 50).round().astype(int)
results


,method,precision_at_50,pages_correct_of_top_50
0,baseline rules,0.24,12
1,logistic regression,0.40,20
2,decision tree,0.54,27
3,random forest,0.74,37


## 6. Self-check

- [x] Named the ML task type: classification (binary probability), consumed downstream as a scoring/
      ranking output — matches how the starter pipeline is actually built.
- [x] Named the target/proxy (`is_declining_label = trend_direction == "down"`), and was explicit that
      it's a proxy, not the ideal future-window label I want to move toward.
- [x] Named the success metric (Precision@50 as primary, average precision / ROC AUC / recall as
      secondary), and tied the choice to the real review-capacity decision from Week 1, not to generic
      accuracy.
- [x] Showed the unit of analysis as a real, executed dataframe — one row = one content item per client
      — and asserted the grain holds (no duplicate `content_id`).
- [x] Sketched what the target column actually looks like next to real feature values.
- [x] Explained why this is an ML problem and not just a rule, backed by the baseline-vs-model numbers
      from this same dataset, not a generic "ML is better" claim.
- [ ] Not yet done, on purpose: retraining the model myself end-to-end (that's the next notebook) — this
      week is framing only, per the brief.
